[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module7/05-attention.ipynb)

# Attention Mechanisms
**Module 7 — Lesson 5 | Estimated time: 30 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU

## Learning Objectives
By the end of this notebook you will be able to:
- Explain attention as a soft lookup mechanism using queries, keys, and values
- Implement scaled dot-product attention in NumPy and PyTorch
- Understand temperature scaling and causal masking
- Build multi-head attention from scratch
- Use `nn.MultiheadAttention` from PyTorch
- Visualise attention weight heatmaps

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 1. Intuition — Attention as a Soft Lookup

Imagine a translator reading a source sentence while generating each target word. Instead of relying on a fixed-length context vector (the RNN bottleneck), attention lets the model **look back at all source tokens** and weigh each by relevance.

Formally, given:
- **Query** `Q` — what we're looking for (e.g. current decoder state)
- **Key** `K` — index of each memory slot (encoder states)
- **Value** `V` — content of each memory slot (encoder states)

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

The `√d_k` scaling prevents the dot-products from growing too large in high dimensions.

In [ ]:
# NumPy implementation — transparent and educational
def scaled_dot_product_attention_np(Q, K, V, mask=None, temperature=1.0):
    """
    Q, K, V: (seq, d_k)
    Returns: (seq, d_v), attention weights (seq, seq)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / (np.sqrt(d_k) * temperature)  # (seq, seq)
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)   # softmax
    output  = weights @ V
    return output, weights

# Small example: seq_len=5, d_k=d_v=4
np.random.seed(7)
seq, dk = 5, 4
Q = np.random.randn(seq, dk)
K = np.random.randn(seq, dk)
V = np.random.randn(seq, dk)

out, w = scaled_dot_product_attention_np(Q, K, V)
print('Output shape:  ', out.shape)
print('Weights (row sums):', w.sum(axis=1).round(6))  # should be all 1.0

plt.figure(figsize=(5, 4))
plt.imshow(w, cmap='Blues')
plt.colorbar()
plt.title('Attention Weight Matrix (5×5)')
plt.xlabel('Key position'); plt.ylabel('Query position')
plt.tight_layout(); plt.show()

## 2. Temperature Scaling

The temperature parameter controls the sharpness of the distribution:
- **Low temperature (< 1)** → peaky, near-argmax attention
- **High temperature (> 1)** → flat, more uniform attention

In [ ]:
temps = [0.25, 0.5, 1.0, 2.0, 5.0]
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for ax, temp in zip(axes, temps):
    _, w_t = scaled_dot_product_attention_np(Q, K, V, temperature=temp)
    im = ax.imshow(w_t, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'T={temp}', fontsize=12)
    ax.set_xlabel('Key'); ax.set_ylabel('Query')
plt.suptitle('Effect of Temperature on Attention Distribution', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 3. Causal Masking

In decoder self-attention, each position may only attend to **previous** positions (no peeking at the future). This is implemented by masking future entries to `-∞` before softmax.

In [ ]:
# Causal (lower-triangular) mask: True means MASKED OUT
causal_mask = np.triu(np.ones((seq, seq), dtype=bool), k=1)
print('Causal mask (True = masked):\n', causal_mask.astype(int))

_, w_causal = scaled_dot_product_attention_np(Q, K, V, mask=causal_mask)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(w, cmap='Blues'); axes[0].set_title('Standard Attention')
axes[1].imshow(w_causal, cmap='Blues'); axes[1].set_title('Causal (Masked) Attention')
for ax in axes:
    ax.set_xlabel('Key'); ax.set_ylabel('Query')
plt.tight_layout(); plt.show()

## 4. Scaled Dot-Product Attention in PyTorch

In [ ]:
def scaled_dot_product_attention_pt(Q, K, V, mask=None):
    """Q, K, V: (batch, heads, seq, d_k)"""
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

# Test (batch=2, 1 head, seq=6, d_k=8)
B, H, S, D = 2, 1, 6, 8
Q_t = torch.randn(B, H, S, D)
K_t = torch.randn(B, H, S, D)
V_t = torch.randn(B, H, S, D)

# Build causal mask for PyTorch
causal_pt = torch.triu(torch.ones(S, S, dtype=torch.bool), diagonal=1)
out_pt, w_pt = scaled_dot_product_attention_pt(Q_t, K_t, V_t, mask=causal_pt)
print('PyTorch attention output:', out_pt.shape)
print('Weights shape:           ', w_pt.shape)
print('First-row weights (should be 1,0,0,...):', w_pt[0, 0, 0].round(3))

## 5. Multi-Head Attention

Instead of a single attention function, use `H` heads in parallel:
1. Project Q, K, V into `H` lower-dimensional subspaces
2. Run scaled dot-product attention in each head
3. Concatenate head outputs
4. Project back to model dimension

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H)\, W^O$$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.h  = num_heads
        self.dk = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def _split_heads(self, x, B, S):
        """(B, S, d_model) -> (B, h, S, dk)"""
        x = x.view(B, S, self.h, self.dk)
        return x.transpose(1, 2)

    def forward(self, q, k, v, mask=None):
        B, S, _ = q.shape
        Q = self._split_heads(self.W_Q(q), B, S)
        K = self._split_heads(self.W_K(k), B, S)
        V = self._split_heads(self.W_V(v), B, S)

        attn_out, weights = scaled_dot_product_attention_pt(Q, K, V, mask)
        # (B, h, S, dk) -> (B, S, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.W_O(attn_out), weights

d_model, num_heads, seq_len = 64, 8, 10
mha = MultiHeadAttention(d_model, num_heads)
x   = torch.randn(2, seq_len, d_model)  # batch=2
out_mha, weights_mha = mha(x, x, x)    # self-attention
print('MHA output shape: ', out_mha.shape)
print('Weights shape:    ', weights_mha.shape)

## 6. nn.MultiheadAttention and Attention Heatmaps

In [ ]:
# PyTorch built-in MHA
mha_pt = nn.MultiheadAttention(embed_dim=64, num_heads=8, batch_first=True)

# Self-attention on a sentence embedding
sentence = "the cat sat on the mat".split()
S = len(sentence)

# Random embeddings (in practice: learned or from model)
torch.manual_seed(0)
embeds = torch.randn(1, S, 64)  # (batch=1, seq, d_model)

with torch.no_grad():
    out_builtin, attn_weights = mha_pt(embeds, embeds, embeds)

# attn_weights: (batch, seq, seq)
w = attn_weights[0].numpy()

plt.figure(figsize=(6, 5))
plt.imshow(w, cmap='YlOrRd')
plt.xticks(range(S), sentence, rotation=45, ha='right')
plt.yticks(range(S), sentence)
plt.colorbar(label='Attention weight')
plt.title('Self-Attention Heatmap\n"the cat sat on the mat"')
plt.tight_layout(); plt.show()
print('Built-in MHA output:', out_builtin.shape)

## Practice Exercises

**Exercise 1 — Cross-Attention**
Modify `MultiHeadAttention` to support cross-attention: pass separate `query` (from decoder) and `key`/`value` (from encoder) tensors of different sequence lengths. Test with `query_seq=5, key_seq=8`.

**Exercise 2 — Attention Entropy**
For each query position, compute the Shannon entropy of its attention weight distribution: `H = -sum(w * log(w + 1e-9))`. High entropy = diffuse attention, low entropy = focused. Plot entropy vs query position for a 20-token sequence.

**Exercise 3 — Visualise Multiple Heads**
Extend the heatmap visualisation to show all 8 heads from `nn.MultiheadAttention` in a 2×4 grid. Use a real sentence (e.g. a Python docstring) and look for head specialisation patterns (e.g. one head focusing on syntax, another on semantics).